In [1]:
!pip install tensorflow==2.19.0
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.9/644.9 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 78.0 MB/s eta 0:00:00
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.4.1
    Uninstalling ml-dtypes-0.4.1:
      Successfully uninstalled ml-dtypes-0.4.1
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.18.0
    Uninstalling tensorboard-2.18.0:
      Successfully uninstalled tensorboard-2.18.0
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.18.0
    Uninstalling tensorflow-2.18.0:
      Successfully uninstalled tensorflow-2.18.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.18.0 requires tensorflow<2.19,>=2.18, but you have tensorflow 2.19.0 w

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
from numpy import array
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import string
import os
import glob
from pickle import dump, load
from time import time
from keras.preprocessing import sequence
from keras.models import Sequential
from keras.layers import LSTM, Embedding, TimeDistributed, Dense, RepeatVector,\
                         Activation, Flatten, Reshape, concatenate, Dropout, BatchNormalization
from keras.optimizers import Adam, RMSprop
from keras.layers import Bidirectional
from keras.layers import add
from keras.applications.inception_v3 import InceptionV3
#from keras.preprocessing import image
from keras.models import Model
from keras import Input, layers
from keras import optimizers
from keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences # Import pad_sequences from keras.utils
from keras.utils import to_categorical
from google.colab import files
#from keras.preprocessing import image
from keras.utils import load_img, img_to_array # Import load_img and img_to_array from keras.utils
import matplotlib.image as mpimg
import shutil
import tensorflow as tf

import gensim.downloader as api
from pickle import dump

# Tải bản nhẹ: 50d hoặc 100d
glove_model = api.load("glove-wiki-gigaword-100")  # 100 chiều, nhẹ và khá tốt
# glove_model = api.load("glove-wiki-gigaword-200")  # nặng hơn nếu bạn muốn
#from google.colab import drive
#drive.mount('/content/drive')
########################### I. PHẦN ĐỌC CÁC THÔNG TIN MÔ TẢ ##################################
path_to_desc = "/content/drive/MyDrive/AITest/Python/dulichbinhthuan.token.txt"

# Đọc file các caption
def load_doc(filename):
        # open the file as read only
        file = open(filename, errors='ignore')
        # read all text
        text = file.read()
        # close the file
        file.close()
        return text

doc = load_doc(path_to_desc)

# Lưu caption dưới dạng key value: id_image : ['caption 1', 'caption 2', 'caption 3',' caption 4', 'caption 5']
def load_descriptions(doc):
        mapping = dict()
        # process lines
        for line in doc.split('\n'):
                # split line by white space
                tokens = line.split()
                if len(line) < 2:
                        continue
                # take the first token as the image id, the rest as the description
                image_id, image_desc = tokens[0], tokens[1:]
                # extract filename from image id
                image_id = image_id.split('.')[0]
                # convert description tokens back to string
                image_desc = ' '.join(image_desc)
                # create the list if needed
                if image_id not in mapping:
                        mapping[image_id] = list()
                # store description
                mapping[image_id].append(image_desc)
        return mapping

descriptions = load_descriptions(doc)

# Hàm tiền xử lý các câu mô tả như: đưa về chữ thường, bỏ dấu, bỏ các chữ số....
def clean_descriptions(descriptions):
        # prepare translation table for removing punctuation
        table = str.maketrans('', '', string.punctuation)
        for key, desc_list in descriptions.items():
                for i in range(len(desc_list)):
                        desc = desc_list[i]
                        # tokenize
                        desc = desc.split()
                        # convert to lower case
                        desc = [word.lower() for word in desc]
                        # remove punctuation from each token
                        desc = [w.translate(table) for w in desc]
                        # remove hanging 's' and 'a'
                        desc = [word for word in desc if len(word)>1]
                        # remove tokens with numbers in them
                        desc = [word for word in desc if word.isalpha()]
                        # store as string
                        desc_list[i] =  ' '.join(desc)

clean_descriptions(descriptions)

# Lưu description xuống file
def save_descriptions(descriptions, filename):
        lines = list()
        for key, desc_list in descriptions.items():
                for desc in desc_list:
                        lines.append(key + ' ' + desc)
        data = '\n'.join(lines)
        file = open(filename, 'w')
        file.write(data)
        file.close()

save_descriptions(descriptions, '/content/drive/MyDrive/AITest/Python/Save/descriptions_python.txt')

################################ II. PHẦN ĐỌC ẢNH  ############################
dirpath = os.getcwd()
path_to_train_id_file = '/content/drive/MyDrive/AITest/Python/dulichbinhthuan.train.txt'
path_to_test_id_file = '/content/drive/MyDrive/AITest/Python/dulichbinhthuan.test.txt'
path_to_image_folder = '/content/drive/MyDrive/AITest/Python/dulichbinhthuan_dataset'

# Lấy id ảnh tương ứng với dữ liệu train, test, dev
def load_set(filename):
        doc = load_doc(filename)
        dataset = list()
        # process line by line
        for line in doc.split('\n'):
                # skip empty lines
                if len(line) < 1:
                        continue
                # get the image identifier
                identifier = line.split('.')[0]
                dataset.append(identifier)
        return set(dataset)

train_id = load_set(path_to_train_id_file)

# 1. -------- Xử lý ảnh dùng để train -----------------------------------------------


# Lấy lấy các ảnh jpg trong thư mục chứa toàn bộ ảnh Flickr

all_train_images_in_folder = glob.glob(path_to_image_folder + '/*.jpg')

# Đọc toàn bộ nội dung file danh sách các file ảnh dùng để train
train_images = set(open(path_to_train_id_file, 'r').read().strip().split('\n'))

# Danh sách ấy sẽ lưu full path vào biến train_img
train_img = []
for images in all_train_images_in_folder : # Duyệt qua tất cả các file trong folder
    if images[len(path_to_image_folder+'/'):] in train_images: # Nếu tên file của nó thuộc training set
        train_img.append(images) # Thì thêm vào danh sách ảnh sẽ dùng để train

# 2. -------- Xử lý ảnh dùng để test (xử lý tương tự) -------------------------------------

test_images = set(open(path_to_test_id_file, 'r').read().strip().split('\n'))
test_img = []

for images in all_train_images_in_folder:
    if images[len(path_to_image_folder+'/'):] in test_images:
        test_img.append(images)

############################ III. PHẦN XỬ LÝ DỮ LIỆU MÔ TẢ ############################

# Hàm đọc mô tả từ file  và Thêm 'startseq', 'endseq' cho chuỗi
def load_clean_descriptions(filename, dataset):
        # load document
        doc = load_doc(filename)
        descriptions = dict()
        for line in doc.split('\n'):
                # split line by white space
                tokens = line.split()
                # split id from description
                image_id, image_desc = tokens[0], tokens[1:]
                # skip images not in the set
                if image_id in dataset:
                        # create list
                        if image_id not in descriptions:
                                descriptions[image_id] = list()
                        # wrap description in tokens
                        desc = 'startseq ' + ' '.join(image_desc) + ' endseq'
                        # store
                        descriptions[image_id].append(desc)
        return descriptions

train_descriptions = load_clean_descriptions('/content/drive/MyDrive/AITest/Python/Save/descriptions_python.txt', train_id)

# Tạo list các training caption
all_train_captions = []
for key, val in train_descriptions.items():
    for cap in val:
        all_train_captions.append(cap)
len(all_train_captions)

# Chỉ lấy các từ xuất hiện trên 10 lần
word_count_threshold = 10
word_counts = {}
nsents = 0
for sent in all_train_captions:
    nsents += 1
    for w in sent.split(' '):
        word_counts[w] = word_counts.get(w, 0) + 1

vocab = [w for w in word_counts if word_counts[w] >= word_count_threshold]

# Tạo từ điển map từ Word sang Index và ngược lại
ixtoword = {}
wordtoix = {}

ix = 1
for w in vocab:
    wordtoix[w] = ix
    ixtoword[ix] = w
    ix += 1

# Tính toán bảng từ vựng
vocab_size = len(ixtoword) + 1 # Thêm 1 cho từ dùng để padding

# Chuyển thành từng dòng
def to_lines(descriptions):
        all_desc = list()
        for key in descriptions.keys():
                [all_desc.append(d) for d in descriptions[key]]
        return all_desc

# Tính toán độ dài nhất của câu mô tả
def max_length(descriptions):
        lines = to_lines(descriptions)
        return max(len(d.split()) for d in lines)

max_length = max_length(train_descriptions)

############################ IV. PHẦN XỬ LÝ ẢNH ĐẦU VÀO ĐỂ ĐƯA VAO MODEL CHÍNH  ############################

# Hàm load ảnh, resize về khích thước mà Inception v3 yêu cầu.
def preprocess(image_path):
    # Resize ảnh về 299x299 làm đầu vào model
    img = load_img(image_path, target_size=(299, 299))
    x = img_to_array(img)
    # Thêm một chiều  nữa
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    return x

# Khởi tạo Model INception v3 để tạo ra feauture cho các ảnh của chúng ta
model = InceptionV3(weights='imagenet')
model_new = Model(model.input, model.layers[-2].output)

# Hàm biến ảnh đầu vào thành vector features (2048, )
def encode(image):
    image = preprocess(image) # preprocess the image
    fea_vec = model_new.predict(image) # Get the encoding vector for the image
    fea_vec = np.reshape(fea_vec, fea_vec.shape[1]) # reshape from (1, 2048) to (2048, )
    return fea_vec

 # Lặp một vòng qua các ảnh train và biến hết thành vector features
encoding_train = {}
for img in train_img:
    print('*TRAIN_IMG* Solving ' + img[len(path_to_image_folder+'/'):])
    encoding_train[img[len(path_to_image_folder+'/'):]] = encode(img)

# Lưu image embedding train lại
with open("/content/drive/MyDrive/AITest/Python/Save/encoded_train_images.pkl", "wb") as encoded_pickle:
    dump(encoding_train, encoded_pickle)

# Lặp một vòng qua các ảnh test và biến hết thành vector features
encoding_test = {}
for img in test_img:
    print('*TEST_IMG* Solving ' + img[len(path_to_image_folder+'/'):])
    encoding_test[img[len(path_to_image_folder+'/'):]] = encode(img)
with open("/content/drive/MyDrive/AITest/Python/Save/encoded_test_images.pkl", "wb") as encoded_pickle:
    dump(encoding_test, encoded_pickle)

train_features = load(open("/content/drive/MyDrive/AITest/Python/Save/encoded_train_images.pkl", "rb"))

############################ V. PHẦN XỬ LÝ MÔ TẢ ĐẦU VÀO ĐỂ ĐƯA VAO MODEL CHÍNH  ############################

# Tải model Glove để embeding word
embedding_dim = 100  # Vì ta dùng bản 100 chiều

# Tạo ma trận embedding rỗng
embedding_matrix = np.zeros((vocab_size, embedding_dim))

# Gán embedding từ GloVe vào từng từ trong wordtoix
for word, i in wordtoix.items():
    if word in glove_model:
        embedding_matrix[i] = glove_model[word]

############################ VI. TẠO MODEL CHÍNH VÀ TIẾN HÀNH TRAIN  ############################

# Tạo model chính

# Nhánh 1. Input image vector
inputs_image = Input(shape=(2048,))
dr_1 = Dropout(0.5)(inputs_image)
fc_1 = Dense(256, activation='relu')(dr_1)

# Nhánh 2. Input câu mô tả
inputs_desc = Input(shape=(max_length,))
emb_1 = Embedding(vocab_size, embedding_dim, mask_zero=True)(inputs_desc)
dr_2 = Dropout(0.5)(emb_1)
lstm_1 = LSTM(256)(dr_2)

# Gộp 2 input
decoder_1 = add([fc_1, lstm_1])
fc_3 = Dense(256, activation='relu')(decoder_1)

# Layer output
outputs = Dense(vocab_size, activation='softmax')(fc_3)
model = Model(inputs=[inputs_image, inputs_desc], outputs=outputs)

# Layer 2 dùng GLOVE Model nên set weight thẳng và không cần train
model.layers[2].set_weights([embedding_matrix])
model.layers[2].trainable = False

# Compile model
model.compile(loss='categorical_crossentropy', optimizer='adam')

#  Hàm sinh dữ liệu train để feed cho Model
def data_generator(descriptions, photos, wordtoix, max_length, num_photos_per_batch):
    X1, X2, y = list(), list(), list()
    n=0
    # loop for ever over images
    while 1:
        for key, desc_list in descriptions.items():
            # Check if the image key exists in the photos dictionary
            if key + '.jpg' not in photos:
                print(f"Skipping image {key}.jpg as it's not found in the photo features.")
                continue

            n+=1
            # retrieve the photo feature
            photo = photos[key+'.jpg']
            for desc in desc_list:
                # encode the sequence
                seq = [wordtoix[word] for word in desc.split(' ') if word in wordtoix]
                # split one sequence into multiple X, y pairs
                for i in range(1, len(seq)):
                    # split into input and output pair
                    in_seq, out_seq = seq[:i], seq[i]
                    # pad input sequence
                    in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
                    # encode output sequence
                    out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]
                    # store
                    X1.append(photo)
                    X2.append(in_seq)
                    y.append(out_seq)
            # yield the batch data
            if n==num_photos_per_batch:
                yield ((array(X1), array(X2)), array(y))  # ✅ đúng chuẩn TensorFlow
                X1, X2, y = list(), list(), list()
                n=0

# Cài đặt tham số train và TRAIN!!!
model.optimizer.lr = 0.0001
epochs = 50 # Số epoch
number_pics_per_batch = 6 # Số lượng ảnh mỗi batch
steps = len(train_descriptions) # Số bước mỗi epoch

for i in range(epochs):
    generator = data_generator(train_descriptions, train_features, wordtoix, max_length, number_pics_per_batch)
    model.fit(generator, epochs=1, steps_per_epoch=steps, verbose=1)

# Sau 10 bước train ta lưu weights
model.save_weights('/content/drive/MyDrive/AITest/Python/Save/model_v2.weights.h5')  # ✅ đúng chuẩn
model.save('/content/drive/MyDrive/AITest/Python/Save/model_v2.keras')  # định dạng mới
model.save('/content/drive/MyDrive/AITest/Python/Save/model_v2.h5')
with open('/content/drive/MyDrive/AITest/Python/Save/wordtoix.pkl', 'wb') as f:
    dump(wordtoix, f)

# Lưu từ điển ixtoword
with open('/content/drive/MyDrive/AITest/Python/Save/ixtoword.pkl', 'wb') as f:
    dump(ixtoword, f)

with open('max_length.pkl', 'wb') as f:
    dump(max_length, f)


# with open('max_length.pkl', 'rb') as f:
#     max_length = load(f)


print("✅ Đã lưu wordtoix.pkl và ixtoword.pkl thành công!")
# Hàm đặt Caption
# Với môi ảnh mới khi test, ta sẽ bắt đầu chuỗi với 'startseq' rồi sau đó cho vào model để dự đoán từ tiếp theo. Ta thêm từ
# vừa được dự đoán vào chuỗi và tiếp tục cho đến khi gặp 'endseq' là kết thúc hoặc cho đến khi chuỗi dài 34 từ.
def setCaption(photo):
    in_text = 'startseq'
    for i in range(max_length):
        sequence = [wordtoix[w] for w in in_text.split() if w in wordtoix]
        sequence = pad_sequences([sequence], maxlen=max_length)
        yhat = model.predict([photo,sequence], verbose=0)
        yhat = np.argmax(yhat)
        word = ixtoword[yhat]
        in_text += ' ' + word
        if word == 'endseq':
            break
    final = in_text.split()
    final = final[1:-1]
    final = ' '.join(final)
    return final

[==================================================] 100.0% 128.1/128.1MB downloaded
96112376/96112376 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
*TRAIN_IMG* Solving dcb1.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
*TRAIN_IMG* Solving SL6.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 417ms/step
*TRAIN_IMG* Solving canh_dong_quat_gio_008.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 403ms/step
*TRAIN_IMG* Solving tb1.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 225ms/step
*TRAIN_IMG* Solving dvtt3.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 465ms/step
*TRAIN_IMG* Solving canh_dong_quat_gio_002.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 263ms/step
*TRAIN_IMG* Solving SL15.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 317ms/step
*TRAIN_IMG* Solving tdt5.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 235ms/step
*TRAIN_IMG* Solving Cong_vien_bien_Doi_Duong_007.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 229ms/step
*TRAIN_IMG* Solving rhklt5.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step
*TRAIN_IMG* Solving dvtt5.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 230ms/step
*TRAIN_IMG* Solving tdt13.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s

KeyboardInterrupt: 

In [ ]:
model.load_weights('/content/drive/MyDrive/AITest/colab/model_v1.keras')

In [ ]:
upload = files.upload()
destination_directory = '/content/drive/MyDrive/AITest/AnhUpLoad/'
for filename, file_content in upload.items():
    destination_path = destination_directory + filename
    with open(destination_path, 'wb') as f:
        f.write(file_content)
    plt.imshow(mpimg.imread(destination_path))
    # Convert thành vector 2048
    image_vector = encode(destination_path).reshape((1,2048))
    print("Thông tin nhận diện hình ảnh:  ",(setCaption(image_vector)))